In [41]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import cv2
import joblib

In [42]:
# Load the dataset
data = pd.read_csv('C:/Users/HP/Desktop/output_pose_data_012215.csv')

In [43]:
# Assuming the CSV file has a 'label' column and the rest are angles
X = data.drop('Label', axis=1)
y = data['Label']

In [44]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
# Initialize and fit the scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [46]:
# Initialize and train the model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

LogisticRegression()

In [47]:
# Save the model and scaler
joblib.dump(model, 'pose_verification_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']

In [48]:
import pandas as pd
import numpy as np
import joblib

In [49]:
# Load the trained model and scaler
model = joblib.load('pose_verification_model.pkl')
scaler = joblib.load('scaler.pkl')

In [56]:
# Example keypoints DataFrame for prediction
keypoints_df = pd.DataFrame({
    'left_shoulder_x': [0.1], 'left_shoulder_y': [0.4],
    'left_elbow_x': [0.2], 'left_elbow_y': [0.5],
    'left_wrist_x': [0.3], 'left_wrist_y': [0.6],
    'right_shoulder_x': [0.1], 'right_shoulder_y': [0.4],
    'right_elbow_x': [0.2], 'right_elbow_y': [0.5],
    'right_wrist_x': [0.3], 'right_wrist_y': [0.6],
})

In [57]:
# Example function to calculate angles
def calculate_angle(a, b, c):
    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.arccos(cosine_angle)
    return np.degrees(angle)

In [58]:
# Function to extract angles from keypoints
def get_angles_from_keypoints(keypoints_df):
    # Calculate angles for left arm
    angles = {
        'left_shoulder_angle': calculate_angle(
            (keypoints_df['left_shoulder_x'][0], keypoints_df['left_shoulder_y'][0]), 
            (keypoints_df['left_elbow_x'][0], keypoints_df['left_elbow_y'][0]), 
            (keypoints_df['left_wrist_x'][0], keypoints_df['left_wrist_y'][0])),
        'left_elbow_angle': calculate_angle(
            (keypoints_df['left_shoulder_x'][0], keypoints_df['left_shoulder_y'][0]), 
            (keypoints_df['left_elbow_x'][0], keypoints_df['left_elbow_y'][0]), 
            (keypoints_df['left_wrist_x'][0], keypoints_df['left_wrist_y'][0])),
        'right_shoulder_angle': calculate_angle(
            (keypoints_df['left_shoulder_x'][0], keypoints_df['left_shoulder_y'][0]), 
            (keypoints_df['left_elbow_x'][0], keypoints_df['left_elbow_y'][0]), 
            (keypoints_df['left_wrist_x'][0], keypoints_df['left_wrist_y'][0])),
        'right_elbow_angle': calculate_angle(
            (keypoints_df['left_shoulder_x'][0], keypoints_df['left_shoulder_y'][0]), 
            (keypoints_df['left_elbow_x'][0], keypoints_df['left_elbow_y'][0]), 
            (keypoints_df['left_wrist_x'][0], keypoints_df['left_wrist_y'][0])),
        'left_knee_angle': calculate_angle(
            (keypoints_df['left_shoulder_x'][0], keypoints_df['left_shoulder_y'][0]), 
            (keypoints_df['left_elbow_x'][0], keypoints_df['left_elbow_y'][0]), 
            (keypoints_df['left_wrist_x'][0], keypoints_df['left_wrist_y'][0])),
        'right_knee_angle': calculate_angle(
            (keypoints_df['left_shoulder_x'][0], keypoints_df['left_shoulder_y'][0]), 
            (keypoints_df['left_elbow_x'][0], keypoints_df['left_elbow_y'][0]), 
            (keypoints_df['left_wrist_x'][0], keypoints_df['left_wrist_y'][0])),
    }
    return angles

In [59]:
# Extract angles from the keypoints DataFrame
angles = get_angles_from_keypoints(keypoints_df)

In [60]:
# Convert angles to DataFrame with correct feature names
angles_df = pd.DataFrame([angles])


In [63]:
import joblib

scaler = joblib.load('scaler.pkl')
# Get and print the expected feature names
expected_feature_names = scaler.feature_names_in_
print(f"Expected Feature Names: {expected_feature_names}")


Expected Feature Names: ['left_elbow_angle' 'right_elbow_angle' 'left_shoulder_angle'
 'right_shoulder_angle' 'left_knee_angle' 'right_knee_angle']


In [64]:
# Rename and reorder the columns to match the expected feature names
angles_df_corrected = angles_df.rename(columns={
    'left_wrist_angle': 'left_elbow_angle',
    'right_wrist_angle': 'right_elbow_angle'
})

In [65]:
# Ensure the order of columns matches the expected order
angles_df_corrected = angles_df_corrected[expected_feature_names]

In [67]:
# Scale the angles using the previously fitted scaler
angles_scaled = scaler.transform(angles_df_corrected)

In [68]:
# Predict using the trained model
pose_correctness = model.predict(angles_scaled)
print(f"Pose Correctness: {'Correct' if pose_correctness == 1 else 'Incorrect'}")

Pose Correctness: Incorrect


In [82]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [83]:
# Load the trained model and scaler
model = joblib.load('pose_verification_model.pkl')
scaler = joblib.load('scaler.pkl')

In [84]:
# Load the image
image_path = 'C:/Users/HP/Desktop/test_1.jpg'
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [85]:
# Initialize MediaPipe pose estimation
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, model_complexity=2)

In [86]:
# Perform pose detection
results = pose.process(image_rgb)

In [87]:
# Extract keypoints
keypoints = results.pose_landmarks.landmark if results.pose_landmarks else None

In [88]:
# Close the MediaPipe instance
pose.close()


In [89]:

if keypoints:
    def calculate_angle(a, b, c):
        a = np.array(a)  # First point
        b = np.array(b)  # Middle point
        c = np.array(c)  # End point
        
        radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
        angle = np.abs(radians * 180.0 / np.pi)
        
        if angle > 180.0:
            angle = 360 - angle
            
        return angle

    # Extract the necessary keypoints for angle calculation
    left_shoulder = [keypoints[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
                     keypoints[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]
    left_elbow = [keypoints[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]
    left_wrist = [keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

    right_shoulder = [keypoints[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,
                      keypoints[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
    right_elbow = [keypoints[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
    right_wrist = [keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

    # Calculate the angles
    left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
    left_shoulder_angle = calculate_angle(left_elbow, left_shoulder, [left_shoulder[0], left_shoulder[1] - 1])
    right_shoulder_angle = calculate_angle(right_elbow, right_shoulder, [right_shoulder[0], right_shoulder[1] - 1])
    
    left_knee = [keypoints[mp_pose.PoseLandmark.LEFT_KNEE.value].x,
                 keypoints[mp_pose.PoseLandmark.LEFT_KNEE.value].y]
    left_hip = [keypoints[mp_pose.PoseLandmark.LEFT_HIP.value].x,
                keypoints[mp_pose.PoseLandmark.LEFT_HIP.value].y]
    left_ankle = [keypoints[mp_pose.PoseLandmark.LEFT_ANKLE.value].x,
                  keypoints[mp_pose.PoseLandmark.LEFT_ANKLE.value].y]

    right_knee = [keypoints[mp_pose.PoseLandmark.RIGHT_KNEE.value].x,
                  keypoints[mp_pose.PoseLandmark.RIGHT_KNEE.value].y]
    right_hip = [keypoints[mp_pose.PoseLandmark.RIGHT_HIP.value].x,
                 keypoints[mp_pose.PoseLandmark.RIGHT_HIP.value].y]
    right_ankle = [keypoints[mp_pose.PoseLandmark.RIGHT_ANKLE.value].x,
                   keypoints[mp_pose.PoseLandmark.RIGHT_ANKLE.value].y]

    left_knee_angle = calculate_angle(left_hip, left_knee, left_ankle)
    right_knee_angle = calculate_angle(right_hip, right_knee, right_ankle)


In [92]:
# Create a DataFrame for the angles
    angles_df = pd.DataFrame([{
        'left_elbow_angle': left_elbow_angle,
        'right_elbow_angle': right_elbow_angle,
        'left_shoulder_angle': left_shoulder_angle,
        'right_shoulder_angle': right_shoulder_angle,
        'left_knee_angle': left_knee_angle,
        'right_knee_angle': right_knee_angle
    }])

    # Ensure the columns are in the correct order
    angles_df = angles_df[['left_elbow_angle', 'right_elbow_angle', 'left_shoulder_angle',
                           'right_shoulder_angle', 'left_knee_angle', 'right_knee_angle']]

    # Scale the angles using the previously fitted scaler
    angles_scaled = scaler.transform(angles_df)

    # Predict using the trained model
    pose_correctness = model.predict(angles_scaled)
    print(f"Pose Correctness: {'Correct' if pose_correctness == 1 else 'Incorrect'}")
else:
    print("No keypoints detected")

IndentationError: unexpected indent (1249774863.py, line 2)